In [ ]:
#!/usr/bin/env -S poetry run python

# Add this to the first cell of your notebook
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv()  # take environment variables from .env


In [2]:
from app.domains.books.schemas.book_filter import BookFilter
from app.db.models import BookModel
from app.domains.books.schemas.utils import QueryBuilder

In [8]:
# Add this to a new cell to start a SQLAlchemy session
from sqlalchemy.ext.asyncio import create_async_engine, async_sessionmaker, AsyncSession
from app.config.settings import settings

print('🎯 Starting SQLAlchemy Session')
print('=' * 50)

# Create async engine using your database settings
async_engine = create_async_engine(settings.postgres.sqlalchemy_url)
AsyncSessionLocal = async_sessionmaker(async_engine, class_=AsyncSession)

print("✅ Session factory ready")

🎯 Starting SQLAlchemy Session
✅ Session factory ready


In [6]:
filters = BookFilter(authors=["King"], min_rating=4.0, limit=3, genre="fiction")
builder = QueryBuilder(BookModel)

# # Option 1: get SQLAlchemy statement (for async execution)
stmt = builder.build(filters)
print(stmt)
print("------------------------\n")
# Option 2: get compiled SQL string (for logging/debugging)
sql_string = builder.build(filters, compile_sql=True)
print(sql_string)


SELECT books.isbn13, books.title, books.authors, books.average_rating, books.num_pages, books.published_year, books.genre, books.categories, books.description 
FROM books 
WHERE (array_position(string_to_array(books.authors, :string_to_array_1), :array_position_1) IS NOT NULL OR lower(books.authors) LIKE lower(:authors_1)) AND books.average_rating >= :average_rating_1 AND books.genre = :genre_1
 LIMIT :param_1
------------------------

SELECT books.isbn13, books.title, books.authors, books.average_rating, books.num_pages, books.published_year, books.genre, books.categories, books.description 
FROM books 
WHERE (array_position(string_to_array(books.authors, ';'), 'King') IS NOT NULL OR lower(books.authors) LIKE lower('%King%')) AND books.average_rating >= 4.0 AND books.genre = 'fiction'
 LIMIT 3


In [7]:
# Test the connection and run your query
async def test_query():
    async with AsyncSessionLocal() as session:
        print("🔌 Connected to database")
        
        # Test your QueryBuilder
        filters = BookFilter(authors=["King"], min_rating=4.0, limit=3, genre="fiction")
        builder = QueryBuilder(BookModel)
        stmt = builder.build(filters)
        
        print(f"📝 Executing query: {stmt}")
        
        try:
            result = await session.execute(stmt)
            rows = result.fetchall()
            
            print(f"📊 Found {len(rows)} results:")
            for row in rows:
                print(f"   Author: {row.authors}, Title: {row.title}, Rating: {row.average_rating}")
                
        except Exception as e:
            print(f"❌ Query failed: {e}")
            
        return rows

# Run the test
results = await test_query()

🔌 Connected to database
📝 Executing query: SELECT books.isbn13, books.title, books.authors, books.average_rating, books.num_pages, books.published_year, books.genre, books.categories, books.description 
FROM books 
WHERE (array_position(string_to_array(books.authors, :string_to_array_1), :array_position_1) IS NOT NULL OR lower(books.authors) LIKE lower(:authors_1)) AND books.average_rating >= :average_rating_1 AND books.genre = :genre_1
 LIMIT :param_1
📊 Found 3 results:
   Author: Joy King, Title: Dirty Little Secrets, Rating: 4.16
   Author: Stephen King;Berni Wrightson;Michele Wrightson, Title: Stephen King's Creepshow, Rating: 4.07
   Author: Karen Kingsbury, Title: Ever After, Rating: 4.32
